# RAG



## RAGとは

RAG（Retrieval-Augmented Generation）は、

> 文書検索（Retrieval）とLLMによる文章生成（Generation）を組み合わせる手法

です。

通常のLLMは学習済み知識だけを使いますが、RAGでは質問時に外部文書を検索し、その内容を根拠として回答します。




## なぜRAGが必要か

LLM単体では、

```text
質問
↓
LLM
↓
回答
```

となります。

数理的には、

$$
P(\text{answer}\mid \text{question})
$$

を計算しています。

そのため、

* 学習後の情報を知らない
* 社内文書を知らない
* PDFを事前学習していない
* ハルシネーションが起こる

という問題があります。





## RAGの基本構造

RAGでは、

```text
質問
↓
文書検索
↓
関連文書取得
↓
LLM
↓
回答
```

となります。

数理的には、

$$ P(\text{answer}\mid\text{question},\text{retrieved\_documents}) $$

です。

つまり、

```text
retrieved_documents
```

を追加条件として回答を生成します。





## RAGの2段階

RAGは通常、

```text
Indexing
Retrieval + Generation
```

の2段階に分かれます。





## 1. Indexing

文書を検索可能な形に変換します。

```text
PDF
↓
Chunk分割
↓
Embedding
↓
Vector DB保存
```

です。

### Chunk分割

長い文書を小さく分割します。

例えば：

```text
論文
↓
paragraph
↓
chunk
```

です。

### Embedding

文章をベクトルへ変換します。

例えば：

```text
「猫」
↓
[0.13, -0.45, ...]
```

のような数値ベクトルになります。

意味が近い文章は、

```text
ベクトル空間で近い
```

という性質があります。

### Vector DB

Embeddingを保存するデータベースです。

例：

* Chroma
* FAISS
* Qdrant
* Pinecone
* Milvus





## 2. Retrieval + Generation

質問時に行う処理です。

### Retrieval

質問：

```text
ARiSEの重複応募制限は？
```

をEmbeddingへ変換します。

```text
質問
↓
Embedding
↓
類似chunk検索
```

を行います。

### Retrieval結果

例えば：

```text
Chunk 1:
研究開発代表者として応募できるのは1件のみ

Chunk 2:
SPReAD代表者はARiSE共同代表者になれない
```

が取得されます。

### Context生成

取得したchunkをまとめます。

```text
<context>

Chunk 1 ...

Chunk 2 ...

</context>
```

### Generation

LLMへ渡します。

```text
質問
+
context
↓
LLM
↓
回答
```

です。




## 全体図

```text
PDF
 ↓
Chunk分割
 ↓
Embedding
 ↓
Vector DB
────────────────────
質問
 ↓
Embedding
 ↓
類似検索
 ↓
関連chunk
 ↓
Context
 ↓
LLM
 ↓
回答
```





## RAGの利点

### 1. 最新情報が使える

LLMを再学習しなくても、

```text
PDF追加
↓
即利用可能
```

です。

例えば：
* 最新のrepository

version upが頻繁に起きるpackageでは特に有効です。

### 2. 社内文書・個人文書を扱える

例えば：

* 研究提案書
* 社内マニュアル
* 契約書
* 論文

など。

### 3. 自分専用の知識ベースを利用できる

LLM単体では一般的な知識に基づいて回答するため、研究グループ内で蓄積された知見や、自分が重要だと考える論文の内容を優先して利用することは難しい。

一方、RAGでは自分で収集した論文や技術資料を検索対象とできるため、自身の研究分野や組織固有の知識に基づいた説明、要約、文書作成を行うことができる。

この点は、NotebookLM や社内文書検索システムが有効な理由の一つでもある。RAGによって、LLMを「一般知識のAI」から「自分専用の知識を活用するAI」へ拡張できる。

### 4. ハルシネーション低減

LLMに

```text
検索結果だけを根拠に答えろ
```

と指示できます。

### 5. 学習不要

通常のFine-tuningは不要です。

```text
文書追加
↓
Embedding作成
↓
検索可能
```

です。

## RAGの弱点

### Chunk依存

検索されなかった情報は使えません。

```text
重要文が検索漏れ
↓
回答失敗
```

となります。

### Long-range dependency

例えば：

```text
1章に定義
10章に結論
```

のような場合、

別chunkになるため検索が難しいことがあります。

### 複雑推論

RAGは基本的に：

```text
検索
+
要約
```

が得意です。

しかし、

* 複数文書を横断した推論
* 因果解析
* 複雑な計画生成

は苦手なことがあります。




## 現在の実用的なRAG

現在よく使われる構成は：

```text
Document
↓
Chunking
↓
Embedding
↓
Vector DB
↓
Retriever
↓
LLM
```

です。

代表的な実装は：

* LangChain
* LlamaIndex
* Haystack

などです。




## 本質

RAGを一言で言うと、

> 必要な文書を検索して、その検索結果をLLMのコンテキストとして与える技術

です。

TransformerやLLMそのものを変更するのではなく、

```text
検索
+
プロンプト注入
```

によって外部知識を利用できるようにしています。



---



## RAGコード全体の概要

現段階で以下のRAGを作成可能なフレームワークが存在します。

| 用途          | 選択肢        |
| ----------- | ---------- |
| 初学者         | LangChain  |
| PDF中心       | LlamaIndex |
| 大企業検索       | Haystack   |
| Prompt最適化研究 | DSPy       |
| Agent付き     | AutoGen    |
| GUI         | Dify       |
| ローカルRAG     | Open WebUI |
| 軽量          | LightRAG   |
| フルスタックRAG   | R2R        |


今回作ったRAGは、langchain + openAI APIを用いて二段階実装してあります。

```text
Step 1: PDF → chroma_db 作成・追加
Step 2: chroma_db + 質問ファイル → 回答生成
```





## Step 1: `build_chroma_db.py`

目的は、

```text
PDFを読み込み
↓
小さいchunkに分割
↓
embedding化
↓
Chroma DBに保存
```

です。



### 1. PDFを読む

```python
loader = PyPDFLoader(str(pdf_path))
docs = loader.load()
```

ここでPDFをページ単位の `Document` に変換します。

各 `Document` は概念的には：

```python
Document(
    page_content="本文...",
    metadata={"source": "...", "page": 0}
)
```

です。



### 2. chunkに分割する

```python
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

splits = splitter.split_documents(docs)
```

PDF全体をそのまま検索対象にすると長すぎるため、小さい単位に分割します。
なお、RecursiveCharacterTextSplitterなどはlangchainの関数です。

例えば：

```text
PDF 1ページ
↓
chunk 1
chunk 2
chunk 3
```

のようになります。

`chunk_overlap` は、chunk間で文脈が切れすぎないように、少し重ねる設定です。



### 3. sourceファイル名からIDを作る

```python
doc_id = make_doc_id(source_name, page, chunk_index)
```

IDは：

```text
sourceファイル名 + page番号 + chunk番号
```

から作っています。

例：

```text
arise_youryo_5.20.pdf::page=10::chunk=2
```

これをハッシュ化してIDにします。

目的は、同じPDFを再登録しても重複しないようにすることです。



### 4. metadataを保存する

```python
doc.metadata["source"] = source_name
doc.metadata["source_path"] = str(pdf_path)
doc.metadata["page"] = page
doc.metadata["chunk_index"] = chunk_index
doc.metadata["doc_id"] = doc_id
```

これにより、検索後に：

```text
どのPDFの
何ページの
どのchunkか
```

を追跡できます。



### 5. embeddingを作る

```python
embeddings = OpenAIEmbeddings(model=embedding_model)
```

embeddingとは、文章を数値ベクトルに変換したものです。

```text
文章
↓
ベクトル
```

意味が近い文章ほど、ベクトル空間で近くなります。



### 6. Chroma DBを開く

```python
vectorstore = Chroma(
    persist_directory=str(db_dir),
    embedding_function=embeddings,
    collection_name=collection_name,
)
```

ここで `chroma_db` を作成または読み込みます。

既存DBがあれば、それに追加できます。



### 7. 既存IDを確認して重複を避ける

```python
existing = vectorstore.get(ids=ids)
existing_ids = set(existing.get("ids", []))
```

すでに登録済みのchunk IDを調べます。

その後：

```python
if doc_id not in existing_ids:
    new_docs.append(doc)
    new_ids.append(doc_id)
```

として、新規chunkだけを追加します。



### 8. Chroma DBへ追加

```python
vectorstore.add_documents(
    documents=new_docs,
    ids=new_ids,
)
```

これでPDFのchunkが `chroma_db` に保存されます。





## Step 2: `ask_rag.py`

目的は、

```text
質問ファイルを読む
↓
Chroma DBから関連chunkを検索
↓
検索結果をcontextとしてLLMへ渡す
↓
回答を生成する
```

です。



### 1. 質問ファイルを読む

```python
questions = load_questions(question_file)
```

`questions.txt` は1行1質問です。

例：

```text
ARiSE申請の重複制限はありますか？
SPReAD申請の重複制限はありますか？
```



### 2. Chroma DBを開く

```python
vectorstore = Chroma(
    persist_directory=str(db_dir),
    embedding_function=embeddings,
    collection_name=collection_name,
)
```

Step 1で作った `chroma_db` を読み込みます。



### 3. Retrieverを作る

```python
retriever = vectorstore.as_retriever(
    search_kwargs={"k": k}
)
```

`retriever` は、質問に近いchunkを探す部品です。

`k=5` なら、関連度の高いchunkを5個取得します。



### 4. 質問ごとに関連chunkを検索する

```python
docs = retriever.invoke(question)
```

ここで質問文をembedding化し、Chroma DB内のchunkと類似度比較します。

流れは：

```text
質問
↓
質問embedding
↓
Chroma DB検索
↓
関連chunk取得
```

です。



### 5. contextを作る

```python
context = make_context(docs)
```

取得したchunkをまとめて、LLMに渡すための `context` を作ります。

概念的には：

```text
<context>
[Document 1]
source: ARiSE.pdf
page: 10
本文...

[Document 2]
source: SPReAD.pdf
page: 5
本文...
</context>
```

になります。



### 6. LLMに質問する

```python
prompt = f"""
1.まず、以下のcontextだけを根拠に質問に日本語で答えてください。
2.もし、1で回答できなかった場合は、「文書中には見つかりませんが、contextから推測します。」と書いてから、contextを元に推測し
てください。


<context>
{context}
</context>

質問:
{question}
"""

response = llm.invoke(prompt)
```

LLMは、検索されたchunkだけを根拠に回答します。

つまり通常のLLM：

```text
P(answer | question)
```

ではなく、RAGでは：

```text
P(answer | question, retrieved_context)
```

になります。



### 7. 回答とsourceを保存する

```python
results.append(
    {
        "question": question,
        "answer": answer,
        "sources": sources,
    }
)
```

`answers.json` には、

```json
{
  "question": "...",
  "answer": "...",
  "sources": [
    {
      "source": "arise_youryo_5.20.pdf",
      "page": 10,
      "content_preview": "..."
    }
  ]
}
```

のように保存されます。




### 重要な点

LLMがPDF全体を読んでいるわけではありません。

毎回、

```text
質問に近いchunkだけ
```

を検索して、promptに入れています。

そのため、RAGの性能は主に：

```text
chunk分割
embedding
retrieverの精度
kの値
prompt設計
```

で決まります。


---



## 文書

ARiSEとSPReADの要項を用いた。



## vector DB

```
$ tree chroma_db/
chroma_db/
├── 542a6eaa-3439-42ce-a454-e75d14d8ef69
│   ├── data_level0.bin
│   ├── header.bin
│   ├── length.bin
│   └── link_lists.bin
└── chroma.sqlite3

1 directory, 5 files
```



## questions.txt

```
ARiSE申請の重複制限はありますか？詳細を教えてください。
SPReAD申請の重複制限はありますか？詳細を教えてください。
ARiSEの研究開発分担者はSPReADの研究代表者になれますか？
```



## answers.txt

    "question": "ARiSEの研究開発分担者はSPReADの研究代表者になれますか？",
    "answer": "文書中には見つかりませんが、contextから推測します。\n\nARiSEの研究開発分担者がSPReADの研究代表者になれるかについて、直接の記述はありません。しかし、Document 1やDocument 2には、SPReADの研究代表者が同時にARiSEの共同代表者を務めることはできない（重複不可）と明記されていますが、ARiSEの研究開発分担者とSPReADの研究代表者の関係については特に制限が示されていません。\n\nしたがって、ARiSEの研究開発分担者がSPReADの研究代表者になることは、重複応募の制限に抵触しないと推測されます。つまり、ARiSEの研究開発分担者はSPReADの研究代表者になることが可能と考えられます。",
    "sources": [
      {
        "source": "arise_youryo_5.20.pdf",
        "page": 44,
        "content_preview": "45 \n3.2.5  重複応募の制限 \n① 研究開発代表者として応募できるのは 本事業全体で1 件のみです。 \n② AI for Science 萌芽的挑戦研究創出事業（SPReAD）の研究代表者が同時にARiSEの共同代\n表者を務めることはできません。 \n③ その他の競争的研究費制度との間に重複応募の制限はありません。 \n④ 提案において、共同研究開発代表者または研究開発分担者として研究開発に参画する場合\nは以下の制限があります。 \na． 研究開発代表者と共同研究開発代表者が互いに入れ替わって、複数件の応募をするこ\nとはできません。 \nb． 研究開発代表者または共同研究開発代表者として応募し"
      },
      {
        "source": "arise_youryo_5.20.pdf",
        "page": 4,
        "content_preview": "の活用先となる研究分野の研究者 （以下、 ドメイン研究者）が協働して研究プロセスの設計から検\n証まで取り組むことが重要です。 \n本事業では、以下のそれぞれの研究タイプにおいて、AI 研究者とドメイン研究者がお互いに知\n見を共有しあい協業することで達成しうる野心的な研究開発目標等を設定し、その達成に向けた挑\n戦的な研究開発提案を募集します。AI研究者とドメイン研究者が「共同代表者」として、提案時に\n共通の「問い」を設定して、その問いを解決するために必要な研究開発計画および体制を提案して\nください。 \nなお、研究開発代表者として応募できるのは、本事業全体で 1 件のみです。AI for Scien"
      },
      {
        "source": "arise_youryo_5.20.pdf",
        "page": 28,
        "content_preview": "29 \nに得てください。 \n特に、国際連携パートナーが共同研究開発代表者を務める場合には、国際連携パートナーが共\n同代表者としての責務（「4.5  採択された共同代表者の責務等」を参照）を負うことができるよ\nう、研究開発代表者や研究開発分担者が所属する国内の研究機関とJSTとの間の委託研究契約書\nに反しない範囲で、研究開発代表者の所属機関と国際連携パートナーの所属機関との間で、国際\n共同研究契約等を締結するなど必要な措置を講ずる必要があります。 \n \n2.2.5  重複応募の制限 \n① 研究開発代表者として応募できるのは 本事業全体で 1 件のみです。 \n② AI for Science 萌芽"
      },
      {
        "source": "SPReAD_youryo.pdf",
        "page": 4,
        "content_preview": "このため、本事業では、研究者等が AI 導入に必要な知見を適時適切に得ながら研究を推進でき\nるよう、 伴走支援体制を構築 ・ 提供いたします。 これにより、 研究計画の具体化、 計算資源の活用、\nデータ整備等における AI 利活用上の留意点の把握を支援し、研究の高度化を促進するとともに、\n分野横断的な連携や実践知の共有につなげます。 \n具体的には下記支援を予定しております。 \n 研究者コミュニティの形成および、当該コミュニティに関連するイベントの企画・運営等 \n 研究実施に際して生じる課題に関する問い合わせ窓口の設置・案内 \n伴走支援の詳細につきましては文部科学省 HP（https://w"
      },
      {
        "source": "arise_youryo_5.20.pdf",
        "page": 104,
        "content_preview": "105 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n AI for Science革新的研究推進事業（ARiSE）の公募選考に関する最新情報は、下記のサイト\nをご参照ください。 \nhttps://www.jst.go.jp/program/arise/koubo.html \n \n【問い合わせ先】 \n※選考経過や採択に関する問い合わせには、一切応じられません  \nお問い合わせは必ず電子メールでお願いします。 \n  \n国立研究開発法人科学技術振興機構（JST） \n 研究プロジェクト推進部 推進第３グループ \n Email: arise.koubo [at] jst.go.jp  \n（"
      }
    ]
  }



## 検索対象の構造を利用する発展例

# RAGの改良（Corpus2Skill）



## flat な similarity search の問題

従来の RAG は、多くの場合：

```text id="v4r1ci"
query
 → embedding
 → 全 document と similarity 計算
 → top-k retrieval
 → LLM
```

という構造を持つ。

数式で書くと：

```math id="kjlwm1"
d^*
=
\arg\max_{d_i \in D}
\mathrm{sim}(q,d_i)
```

であり、

* query embedding
* document embedding

の類似度だけで retrieval を行う。



## なぜ問題なのか

### 1. corpus 構造が見えない

flat retrieval では：

```text id="yd1myt"
LLM ← top-k passages
```

だけが渡される。

つまり LLM は：

* corpus全体のtopic構造
* 他にどんな領域があるか
* retrieval漏れがあるか

を知らない。〔Sun2026Corpus2Skill〕 

論文ではこれを：

> “the LLM never sees the forest for the trees”

と表現している。〔Sun2026Corpus2Skill〕 

### 2. embedding similarity は局所一致しか見ない

embedding retrieval は：

```text id="l5r8gx"
単語
短距離意味類似
```

に強い。

しかし：

```text id="slhvsk"
business type変更
 → payment policy
 → support workflow
```

のような multi-hop 的関係は弱い。〔Sun2026Corpus2Skill〕 

つまり：

```text id="llr7u8"
query と document の
直接類似
```

しか見ない。

### 3. retrieval miss を認識できない

flat retrieval では：

```text id="qf6nsp"
top-k に入らなかった document
```

は完全に invisible。

LLM は：

* 他branchの存在
* 他topicの存在

を知らない。〔Sun2026Corpus2Skill〕 

そのため：

* backtracking
* exploration
* branch変更

ができない。

### 4. retrieval が black box

従来RAGでは：

```math id="f36a1i"
D_q = f(q,D)
```

であり、

retrieval function (f) が：

* なぜそのdocumentを選んだか
* 他候補が何か

をLLMに見せない。〔Sun2026Corpus2Skill〕 



## Sun2026Corpus2Skill の核心

この問題に対して論文は：

> 「最初から全文書に対して flat な similarity search をするのでなく、
> 階層的に探索空間を絞り込め」

という設計を提案する。〔Sun2026Corpus2Skill〕 



## Corpus2Skill の考え方

まず corpus を：

```text id="ztz5t8"
commerce
marketing
billing
support
```

のような topic hierarchy に変換する。

さらに：

```text id="e91h90"
billing
 ├ payments
 ├ invoice
 ├ refund
```

のように細分化する。

つまり：

```text id="az4gxp"
coarse
 → medium
 → fine
```

という探索空間を事前に構築する。〔Sun2026Corpus2Skill〕 



## retrieval から navigation へ

従来：

```text id="j3v2pk"
query
 → embedding search
 → top-k
```

Corpus2Skill：

```text id="u8hmn9"
query
 → topic選択
 → subgroup選択
 → leaf選択
 → document取得
```

つまり：



## retrieval

ではなく



## navigation

を行う。〔Sun2026Corpus2Skill〕 



## 数理的な違い

### flat retrieval

従来：

```math id="8rv10k"
P(d|q)
```

を直接推定。

### Corpus2Skill

階層分解：

```math id="7k2ns4"
P(d|q)
=
P(d|s_L)
P(s_L|s_{L-1})
\cdots
P(s_1|q)
```

つまり：

```text id="8zw3hl"
query
 → topic
 → subtopic
 → document
```

という hierarchical routing を行う。



## attention 的に見ると

従来：

```math id="84g1dg"
Attention(q, D)
```

を全文書へ直接行う。

Corpus2Skill：

```math id="7tvf73"
Attention(q, topics)
→
Attention(q, subtopics)
→
Attention(q, documents)
```

つまり：



## hierarchical attention

である。



## Progressive Disclosure

論文の重要概念。〔Sun2026Corpus2Skill〕 

LLM は最初から全文書を読まない。


```text id="7kffnq"
overview
 → narrowing
 → detailed evidence
```

を行う。



## なぜこれが強いのか

論文の主張：



## LLMは overview があると探索できる

ことである。〔Sun2026Corpus2Skill〕 

つまり：

```text id="jlwm7k"
flat retrieval
 = 局所passageのみ
```

に対し、

```text id="0knn9f"
Corpus2Skill
 = corpus全体構造を見ながら探索
```

できる。

そのため：

* branch比較
* backtracking
* cross-branch synthesis

が可能になる。




## Corpus2Skill の本質

Corpus2Skill は：
hierarchyそのものを
reasoning space にする

つまり：

```text id="12wfhv"
hierarchy
 = memory structure
 = navigation structure
 = reasoning structure
```

として扱う。



## 論文の位置づけ

この論文は：

```text id="w9n08w"
flat retrieval
 ↓
hierarchical retrieval
 ↓
agentic navigation
```

という流れの重要論文である。




## 実験結果

| Method       |    F1 |  BLEU | ROUGE-1 | ROUGE-2 | Factuality | Context Recall |
| ------------ | ----: | ----: | ------: | ------: | ---------: | -------------: |
| BM25         | 0.342 | 0.060 |   0.364 |   0.119 |      0.470 |          0.386 |
| Dense        | 0.363 | 0.074 |   0.382 |   0.141 |      0.536 |          0.450 |
| Hybrid       | 0.360 | 0.071 |   0.380 |   0.137 |      0.524 |          0.410 |
| RAPTOR       | 0.389 | 0.109 |   0.406 |   0.189 |      0.675 |          0.616 |
| Agentic      | 0.381 | 0.098 |   0.395 |   0.183 |      0.719 |          0.498 |
| Corpus2Skill | 0.468 | 0.143 |   0.483 |   0.238 |      0.739 |          0.673 |




## 文献

〔Sun2026Corpus2Skill〕
[Sun2026Corpus2Skill] Sun, Y., Wei, P., & Hsieh, L. B., Don’t Retrieve, Navigate: Distilling Enterprise Knowledge into Navigable Agent Skills for QA and RAG, arXiv:2604.14572v2 (2026). DOI: arXiv:2604.14572.


## 実習

1. [PDF を Chroma に登録する](8300.1101.RAG.build_chroma_db.ipynb)
2. [文書を検索して回答を生成する](8300.1201.RAG.ask_rag.ipynb)

文献をノードと関係で整理して検索する方法は [Graph RAG](8300.0160.graph_RAG.ipynb) で扱います。